# Questão 2 - Schemas

## Cenário

Você foi informado que a empresa que fornece os serviços do ERP não permite conexão direta com o banco de dados e a única forma de se obter os dados é através dos CSVs já fornecidos.

Para realizar as próximas etapas precisaremos desses dados carregados em algum banco de dados e, para isso, precisamos definir o schema desse banco primeiro.

Sua missão é desenvolver um código que detecta as colunas de cada tabela a partir dos CSV e cria um arquivo final .sql com as instruções de criação de cada tabela.

## Premissas obrigatórias

- Considere todos os CSV como arquivos de fonte.
- Utilize obrigatoriamente Python 3.
- Utilize somente bibliotecas padrão do Python 3 (csv, os, datetime e etc.) e python puro. Soluções que utilizarem bibliotecas como pandas, dask, polars serão desconsideradas.
- Considere o banco de destino como sendo um PostgreSQL.
## Tarefa: 
- Crie um script python que possa ler os CSV de um diretório e gere um único arquivo de saída (schema.sql) com as instruções de criação de uma tabela para cada arquivo CSV usando somente bibliotecas de acordo com as instruções anteriores.

## Raciocínio da criação do schema

Foi desenvolvido um processo automatizado para gerar o schema PostgreSQL a partir dos 24 arquivos CSV fornecidos. A abordagem foi adotada para reduzir trabalho manual, evitar inconsistências entre tabelas e permitir a reprodução do processo caso novos arquivos sejam adicionados.

Inicialmente, os nomes dos arquivos e das colunas foram normalizados para um padrão seguro para SQL. Essa etapa evita problemas causados por espaços, acentos ou caracteres especiais.

Em seguida, uma amostra de cada arquivo foi analisada para inferir tipos de dados compatíveis com PostgreSQL. 
Para definir os tipos das colunas, foram analisadas as primeiras linhas de cada arquivo. Essa leitura permitiu diferenciar campos numéricos, datas e textos sem carregar todo o arquivo durante a criação do schema.

Foram considerados tipos como `BIGINT`, `NUMERIC`, `DATE`, `TIMESTAMP`, `BOOLEAN` e `TEXT`. Quando foram encontrados valores de tipos diferentes em uma mesma coluna, foi escolhido o tipo mais seguro para preservar os dados, priorizando `TEXT` em casos de ambiguidade.

O limite de 1.000 linhas foi definido como parâmetro técnico e pode ser ajustado. Em caso de formatos mistos ou ambíguos, foi utilizado `TEXT` para reduzir o risco de perda de informação ou falha no carregamento.

Por fim, foi gerado um comando `CREATE TABLE IF NOT EXISTS` para cada CSV. O uso de `IF NOT EXISTS` permite executar o arquivo novamente sem falhar caso uma tabela já exista.

Essa estrutura separa a definição do banco da etapa de carregamento e cria uma base reproduzível para as análises posteriores.

In [2]:
# ============================================================
# IMPORTAÇÃO DE BIBLIOTECAS
# ============================================================

# csv: leitura de arquivos CSV e identificação de delimitadores.
import csv

# os: navegação por arquivos e diretórios.
import os

# re: uso de expressões regulares para validação e normalização.
import re

# sys: leitura de argumentos informados ao executar o script.
import sys

# unicodedata: remoção de acentos para gerar identificadores SQL seguros.
import unicodedata

# date e datetime: validação de valores de data e data/hora.
from datetime import date, datetime


# ============================================================
# ETAPA 1 — NORMALIZAÇÃO DE NOMES DE TABELAS E COLUNAS
# ============================================================

def normalizar_identificador(nome, prefixo="col"):
    """
    Converte nomes de tabelas e colunas para um formato seguro em SQL.

    Exemplo:
    'Preço Médio (R$)' -> 'preco_medio_r'
    """
    # Remove acentos e caracteres que não possuem equivalente ASCII.
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ascii", "ignore").decode("ascii")

    # Remove espaços no início/fim e padroniza em letras minúsculas.
    nome = nome.strip().lower()

    # Substitui caracteres não permitidos por sublinhado.
    nome = re.sub(r"[^a-z0-9_]+", "_", nome)

    # Evita sublinhados repetidos e remove sublinhados nas extremidades.
    nome = re.sub(r"_+", "_", nome).strip("_")

    # Define um nome padrão caso o cabeçalho esteja vazio.
    if not nome:
        nome = prefixo

    # Evita identificadores SQL iniciados por número.
    if nome[0].isdigit():
        nome = f"{prefixo}_{nome}"

    return nome


# ============================================================
# ETAPA 2 — IDENTIFICAÇÃO DO DELIMITADOR DO CSV
# ============================================================

def identificar_dialeto(caminho_csv):
    """
    Identifica delimitadores comuns em arquivos CSV.

    São testados: vírgula, ponto e vírgula, barra vertical e tabulação.
    """
    # Lê uma amostra inicial do arquivo para identificar sua estrutura.
    with open(caminho_csv, "r", encoding="utf-8-sig", newline="") as arquivo:
        amostra = arquivo.read(4096)

    try:
        # Tenta identificar automaticamente o delimitador utilizado.
        return csv.Sniffer().sniff(amostra, delimiters=",;|\t")

    except csv.Error:
        # Usa vírgula como padrão se a identificação automática falhar.
        return csv.excel


# ============================================================
# ETAPA 3 — IDENTIFICAÇÃO DO TIPO DE CADA VALOR
# ============================================================

def identificar_tipo(valor):
    """
    Classifica um valor em um tipo compatível com PostgreSQL.

    Possíveis retornos:
    BOOLEAN, BIGINT, NUMERIC, DATE, TIMESTAMP, TEXT ou None.
    """
    # Remove espaços desnecessários antes da análise.
    valor = valor.strip()

    # Valores vazios não participam da inferência de tipo.
    if valor == "":
        return None

    # Padroniza para validar valores booleanos.
    valor_minusculo = valor.lower()

    # Identifica valores booleanos.
    if valor_minusculo in {"true", "false"}:
        return "BOOLEAN"

    # Identifica números inteiros positivos ou negativos.
    if re.fullmatch(r"[+-]?\d+", valor):

        # Preserva como texto números iniciados por zero,
        # como CEPs, códigos e identificadores.
        if re.fullmatch(r"0\d+", valor):
            return "TEXT"

        return "BIGINT"

    # Identifica números decimais, com ponto ou vírgula.
    if re.fullmatch(r"[+-]?(\d+\.\d+|\d+,\d+|\d+)([eE][+-]?\d+)?", valor):
        return "NUMERIC"

    # Tenta interpretar o valor como data no padrão ISO.
    try:
        date.fromisoformat(valor)
        return "DATE"
    except ValueError:
        pass

    # Tenta interpretar o valor como data e hora.
    try:
        datetime.fromisoformat(valor.replace("Z", "+00:00"))
        return "TIMESTAMP"
    except ValueError:
        pass

    # Valores que não se encaixam nas regras anteriores são texto.
    return "TEXT"


# ============================================================
# ETAPA 4 — DEFINIÇÃO DO TIPO FINAL DE CADA COLUNA
# ============================================================

def definir_tipo_final(tipos_encontrados):
    """
    Define o tipo SQL final da coluna com base nos tipos
    identificados na amostra de valores.
    """
    # Remove tipos repetidos da lista analisada.
    tipos = set(tipos_encontrados)

    # Colunas sem valores preenchidos são criadas como texto.
    if not tipos:
        return "TEXT"

    # Se houver ao menos um texto, a coluna será texto para evitar perda de dados.
    if "TEXT" in tipos:
        return "TEXT"

    # Colunas exclusivamente booleanas.
    if tipos == {"BOOLEAN"}:
        return "BOOLEAN"

    # Colunas exclusivamente inteiras.
    if tipos.issubset({"BIGINT"}):
        return "BIGINT"

    # Mistura de inteiro e decimal deve ser armazenada como número decimal.
    if tipos.issubset({"BIGINT", "NUMERIC"}):
        return "NUMERIC"

    # Colunas exclusivamente de data.
    if tipos.issubset({"DATE"}):
        return "DATE"

    # Mistura de data e data/hora deve ser armazenada como TIMESTAMP.
    if tipos.issubset({"DATE", "TIMESTAMP"}):
        return "TIMESTAMP"

    # Como proteção, utiliza texto para combinações não previstas.
    return "TEXT"


# ============================================================
# ETAPA 5 — LEITURA DO CABEÇALHO E INFERÊNCIA DOS TIPOS
# ============================================================

def obter_colunas_e_tipos(caminho_csv, limite_amostra=1000):
    """
    Lê o cabeçalho e uma amostra do CSV para inferir os tipos
    das colunas que serão criadas no PostgreSQL.
    """
    # Identifica o delimitador antes de ler o arquivo.
    dialeto = identificar_dialeto(caminho_csv)

    with open(caminho_csv, "r", encoding="utf-8-sig", newline="") as arquivo:
        leitor = csv.reader(arquivo, dialeto)

        try:
            # Lê a primeira linha do arquivo como cabeçalho.
            cabecalho_original = next(leitor)
        except StopIteration:
            # Retorna listas vazias quando o CSV não possui conteúdo.
            return [], []

        colunas = []
        nomes_utilizados = set()

        # Normaliza cada nome de coluna do cabeçalho.
        for indice, nome in enumerate(cabecalho_original, start=1):
            coluna = normalizar_identificador(
                nome,
                prefixo=f"coluna_{indice}"
            )

            # Evita nomes repetidos após a normalização.
            nome_base = coluna
            contador = 2

            while coluna in nomes_utilizados:
                coluna = f"{nome_base}_{contador}"
                contador += 1

            nomes_utilizados.add(coluna)
            colunas.append(coluna)

        # Cria uma lista para registrar os tipos encontrados em cada coluna.
        tipos_por_coluna = [[] for _ in colunas]

        # Analisa apenas a quantidade definida no limite de amostra.
        for indice_linha, linha in enumerate(leitor):
            if indice_linha >= limite_amostra:
                break

            # Avalia cada valor presente na linha.
            for indice_coluna in range(len(colunas)):

                # Trata linhas com quantidade menor de valores que o cabeçalho.
                valor = (
                    linha[indice_coluna]
                    if indice_coluna < len(linha)
                    else ""
                )

                tipo = identificar_tipo(valor)

                # Valores vazios não influenciam o tipo final da coluna.
                if tipo is not None:
                    tipos_por_coluna[indice_coluna].append(tipo)

    # Define o tipo SQL final de cada coluna a partir da amostra.
    tipos_finais = [
        definir_tipo_final(tipos)
        for tipos in tipos_por_coluna
    ]

    return colunas, tipos_finais


# ============================================================
# ETAPA 6 — GERAÇÃO DO ARQUIVO schema.sql
# ============================================================

def gerar_schema(diretorio_entrada, arquivo_saida):
    """
    Gera comandos CREATE TABLE para todos os CSVs encontrados
    no diretório informado.
    """
    # Localiza e ordena todos os arquivos CSV do diretório.
    arquivos_csv = sorted(
        arquivo
        for arquivo in os.listdir(diretorio_entrada)
        if arquivo.lower().endswith(".csv")
    )

    # Interrompe a execução caso nenhum CSV seja localizado.
    if not arquivos_csv:
        raise FileNotFoundError(
            "Nenhum arquivo CSV foi encontrado no diretório."
        )

    # Inicia o arquivo SQL com comentários de documentação.
    comandos = [
        "-- Schema gerado automaticamente a partir dos arquivos CSV",
        "-- Banco de destino: PostgreSQL",
        ""
    ]

    # Processa cada arquivo CSV individualmente.
    for arquivo_csv in arquivos_csv:
        caminho_csv = os.path.join(diretorio_entrada, arquivo_csv)

        # Usa o nome do arquivo como nome base da tabela.
        nome_tabela = os.path.splitext(arquivo_csv)[0]

        # Normaliza o nome para garantir compatibilidade com SQL.
        nome_tabela = normalizar_identificador(
            nome_tabela,
            prefixo="tabela"
        )

        # Obtém nomes de colunas e tipos inferidos pela amostra.
        colunas, tipos = obter_colunas_e_tipos(caminho_csv)

        # Ignora arquivos CSV sem cabeçalho ou conteúdo.
        if not colunas:
            print(f"Aviso: {arquivo_csv} está vazio e não foi incluído.")
            continue

        # Monta cada definição de coluna no formato: nome tipo.
        definicoes_colunas = [
            f"    {coluna} {tipo}"
            for coluna, tipo in zip(colunas, tipos)
        ]

        # Cria o comando SQL para a tabela atual.
        comando = (
            f"-- Fonte: {arquivo_csv}\n"
            f"CREATE TABLE IF NOT EXISTS {nome_tabela} (\n"
            + ",\n".join(definicoes_colunas)
            + "\n);\n"
        )

        comandos.append(comando)

    # Grava todos os comandos SQL no arquivo de saída.
    with open(arquivo_saida, "w", encoding="utf-8") as arquivo:
        arquivo.write("\n".join(comandos))

    # Apresenta um resumo da execução no terminal.
    print(f"Schema gerado com sucesso: {arquivo_saida}")
    print(f"Tabelas processadas: {len(arquivos_csv)}")


# ============================================================
# ETAPA 7 — EXECUÇÃO NO NOTEBOOK
# ============================================================

# Usa a pasta atual do notebook como diretório de entrada.
diretorio = "."

# Define o nome do arquivo SQL que será gerado.
saida = "schema.sql"

# Executa a geração do schema.
gerar_schema(diretorio, saida)

Schema gerado com sucesso: schema.sql
Tabelas processadas: 24
